In [1]:
!pip install -q transformers peft accelerate bitsandbytes trl
import os
import tensorflow as tf
from datasets import load_dataset
from transformers import BitsAndBytesConfig
import torch
from peft import LoraConfig, get_peft_model



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requir

2026-05-28 17:43:56.411024: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779990236.628187      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779990236.690210      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779990237.207816      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779990237.207864      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779990237.207867      58 computation_placer.cc:177] computation placer alr

In [5]:

dataset = load_dataset(
    "teknium/OpenHermes-2.5",
    split="train[:5]",
    token=os.getenv("HF_TOKEN")
)
print("dataset loaded")

dataset loaded


In [6]:
def format_data(conversation_example):
   
    flag = False
    string = ''
    for each_interaction in each_conversation['conversations']:
        if(each_interaction['from'] == 'human'and not (flag)):
            string+="Instruction "+each_interaction['value']
            flag = True
            
        elif(flag and each_interaction['from'] != 'human'):
            string+="Responce "+ each_interaction['value']
            flag = False
    return string
all_formatted_conversation = []
for each_conversation in dataset:
    one_formatted_conversation = format_data(each_conversation['conversations'])
    all_formatted_conversation.append(one_formatted_conversation)
print("Data formatted")

Data formatted


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer

model_name = "Qwen/Qwen2.5-3B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("tokenizer loaded")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer loaded


In [9]:
def tokenize_function(example):
    tokens = tokenizer(
        example,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = [tokenize_function(x) for x in all_formatted_conversation]

print("data tokenized")
# input_ids = [x["input_ids"] for x in tokenized_dataset]
# attention_masks = [x["attention_mask"] for x in tokenized_dataset]

# train_dataset = tf.data.Dataset.from_tensor_slices({
#     "input_ids": input_ids,
#     "attention_mask": attention_masks,
# })

# train_dataset = train_dataset.batch(4)

data tokenized


In [10]:

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("model loaded")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597
model loaded


In [11]:

training_args = TrainingArguments(
    output_dir="./qwen-lora-output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
print("starting training")
trainer.train()
print("trained")

starting training


Step,Training Loss
10,11.458963
20,11.539291
30,9.420690


trained
